In [1]:
import os
import qsprpred
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch

from qsprpred.data import QSPRDataset, RandomSplit
from qsprpred.data.descriptors.fingerprints import MorganFP
import pandas as pd
from qsprpred.data.descriptors.sets import RDKitDescs
from MolEval import MolEmb 

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. cannot import name 'DMPNN' from 'deepchem.models.torch_models' (/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/deepchem/models/torch_models/__init__.py)
Skipped loading modules with pytorch-lightning dependency, missing a dependency. No module named 'lightning'
Skipped loading some Jax models, missing a dependency. No module named 'jax'
Skipped loading some PyTorch models, missing a dependency. No module named 'tensorflow'


In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold
from qsprpred.data.descriptors.sets import RDKitDescs

def load_datasets(path):
    dataset = QSPRDataset.fromTableFile(
    filename=path,
    store_dir="dataset_outputs/A2AR/data",
    name="A2ARDataset",
    target_props=[{"name": "Y", "task": "SINGLECLASS", "th": [0.5]}],
    random_state=42,
    smiles_col = 'Drug',
    sep=','
    )
    #dataset.prepareDataset(
    #feature_calculators=[MorganFP(radius=2, nBits=1024)],
    #recalculate_features=True,
    #shuffle=False
    #)
    #from qsprpred.data.descriptors.sets import RDKitDescs
    
    #rdkit_descs = RDKitDescs()
    
    #dataset.addDescriptors([rdkit_descs])
    
    #dataset.descriptorSets
    return dataset

class Dataset_creator():
    def __init__(self, model_names=['RoBERTa_ZINC'], corr_thrsh= 0.95):
        self.variance = VarianceThreshold(threshold=0.0)
        self.model_names = model_names
        self.corr_thrsh = corr_thrsh
        self.selected_indices = None
        
    def fit_transform(self, dataset):
        dataset.prepareDataset(feature_calculators=[MorganFP(radius=2, nBits=4096)], recalculate_features=True, shuffle=False)
        rdkit_descs = RDKitDescs()
        dataset.addDescriptors([rdkit_descs])
        display(dataset.X.shape)
        
        dataset.descriptorSets
        dataset_embs = self.create_embs(dataset)
        dataset_embs.index = dataset.X.index
        dataset.X = pd.concat([dataset.X, dataset_embs], axis=1)
        display(dataset.X.shape)
        
        dataset_no_var_np = self.variance.fit_transform(dataset.X)
        mask = self.variance.get_support()
        dataset_without_no_var = pd.DataFrame(
            dataset_no_var_np,
            columns=dataset.X.columns[mask],
            index=dataset.X.index)
        display(dataset_without_no_var.shape)

        
        dataset_without_high_corr = self.high_correlation(dataset_without_no_var)
        display(dataset_without_high_corr.shape)
        
        dataset.X = dataset_without_high_corr
        return dataset
        
    def transform(self, dataset):
        dataset.prepareDataset(feature_calculators=[MorganFP(radius=2, nBits=4096)], recalculate_features=True, shuffle=False)
        rdkit_descs = RDKitDescs()
        dataset.addDescriptors([rdkit_descs])
        display(dataset.X.shape)

        dataset.descriptorSets
        dataset_embs = self.create_embs(dataset)
        dataset_embs.index = dataset.X.index
        dataset.X = pd.concat([dataset.X, dataset_embs], axis=1)
        display(dataset.X.shape)

        dataset_no_var_np = self.variance.transform(dataset.X)
        mask = self.variance.get_support()
        dataset_without_no_var = pd.DataFrame(
            dataset_no_var_np,
            columns=dataset.X.columns[mask],
            index=dataset.X.index)
        display(dataset_without_no_var.shape)

        dataset_without_high_corr = dataset_without_no_var[self.selected_indices]
        display(dataset_without_high_corr.shape)

        dataset.X = dataset_without_high_corr
        return dataset

    def high_correlation(self, df: pd.DataFrame):
        corr_matrix = df.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        
        to_drop = [column for column in upper.columns if any(upper[column] > self.corr_thrsh)]
        self.selected_indices = df.columns.difference(to_drop)
        
        return df[self.selected_indices]

        
    def create_embs(self, dataset):
        dataset.df["SMILES"] = dataset.df["Drug"]
        final_emb = pd.DataFrame()
        for model_name in self.model_names:
            extractor = MolEmb.EmbeddingExtractor(model_name=model_name, df=dataset.df)
            new_emb, dataset.df = extractor.get_embeddings()
            display(type(new_emb))
            if final_emb.empty:
                final_emb = new_emb
            else:
                final_emb = pd.concat([final_emb, new_emb], axis=1)
        final_emb.columns = final_emb.columns.astype(str)
        display(final_emb)
        return final_emb
    


In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X1_all = load_datasets("A2AR/data/a2ar_train_1")

X2_all = load_datasets("A2AR/data/a2ar_val_1")

X3_all = load_datasets("A2AR/data/a2ar_test_1")

In [4]:
cls = Dataset_creator(['Mol2Vec', 'RoBERTa_ZINC'])

In [5]:
X1_all = cls.fit_transform(X1_all)
X2_all = cls.transform(X2_all)
X3_all = cls.transform(X3_all)

(2407, 4306)

pandas.core.frame.DataFrame

Some weights of RobertaModel were not initialized from the model checkpoint at entropy/roberta_zinc_480m and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


pandas.core.frame.DataFrame

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,3.031286,-3.454808,-2.263860,6.271306,2.995978,0.981784,-10.757834,-1.355217,6.775432,-1.027848,...,0.016533,0.195609,-0.674374,-0.065700,0.370229,-0.326461,0.348396,-0.086251,0.064743,-0.962521
1,4.719957,-6.825691,-5.275553,7.427641,2.117555,-4.409114,-20.588449,-4.263409,9.039432,1.124004,...,-0.117531,0.066307,0.073473,0.053947,-0.011456,0.248302,0.882871,-0.134034,0.101105,-0.926051
2,4.035905,-2.917027,-3.177843,4.870048,0.714673,0.467616,-10.627448,-1.463339,2.508328,3.341819,...,-0.051249,0.019601,-0.234947,-0.663630,-0.225576,0.667409,0.739919,-0.373016,-0.551484,-0.496429
3,5.960171,-7.855495,-4.323918,8.447305,2.946618,-2.937841,-19.243912,-5.039819,6.695743,1.658576,...,-0.224458,0.155745,-0.224500,0.159243,-0.185103,-0.191579,0.361649,0.198005,-0.138387,-0.536161
4,4.958643,-7.047005,-2.875748,7.948713,0.880517,1.066714,-11.513125,-1.860976,5.726758,0.257126,...,0.066663,0.252491,-0.544621,-0.063162,0.201588,-0.360610,0.267816,-0.085702,-0.012495,-0.989619
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2402,2.653754,-7.146047,-2.449787,9.172974,1.035106,0.775873,-14.914745,-2.719624,8.021161,2.170897,...,0.448256,0.491986,-0.357603,-0.081183,-0.278233,-0.135188,0.569269,-0.283329,-0.071396,-0.738980
2403,1.443035,-5.894413,-4.667930,10.747350,-1.742077,-1.549439,-15.353065,-1.347566,9.008447,5.855970,...,0.128226,0.159901,-0.013588,-0.477770,0.085022,0.110087,0.779071,-0.281327,0.103782,-0.730889
2404,3.461942,-5.417378,-3.468525,7.684143,1.368830,-0.217005,-13.005549,-3.189668,8.138423,2.356131,...,0.448488,-0.177408,-0.629834,-0.152293,-0.186566,0.062021,0.700484,-0.078713,0.170892,-0.848403
2405,2.946900,-4.950596,-2.974077,8.789262,-0.511377,0.093030,-12.781496,-2.486715,6.638522,1.301827,...,0.141541,0.003172,-0.304752,-0.335208,0.216789,-0.193689,0.674777,-0.341150,0.009661,-0.460586


...

pandas.core.frame.DataFrame

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,3.350501,-1.980245,-3.243645,4.621302,-2.423914,-0.357021,-12.652489,0.289971,5.873908,4.563121,...,0.066493,0.313761,-0.687266,-0.086003,0.015799,0.499730,0.417170,-0.433060,0.025140,-0.010341
1,3.714069,-5.852489,-2.354247,9.809155,-0.262454,1.374667,-12.131661,-2.852122,5.135067,0.854150,...,0.427338,-0.119034,-0.220593,-0.327747,0.018900,-0.180831,0.066138,-0.359251,0.187227,-0.331075
2,2.676739,-4.529388,-2.503095,8.664835,-1.422678,1.718229,-10.841211,-0.888626,5.250077,1.132984,...,0.361555,-0.109810,-0.367811,0.251796,-0.052785,0.133834,0.889266,-0.269218,0.163290,-0.007741
3,3.081486,-8.096527,-2.389479,9.637930,1.337607,0.428647,-15.852401,-2.957059,7.091984,2.542176,...,0.212281,0.368771,0.153928,-0.228844,0.012988,-0.085077,0.429632,-0.064018,0.113950,-0.692819
4,2.528946,-2.400130,-2.897563,7.094854,-0.478954,0.433469,-10.049082,-1.011271,4.869930,3.022278,...,0.089385,-0.039821,-0.385214,-0.412184,0.306683,0.209005,0.888776,-0.416939,-0.295426,-0.088778
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
859,4.397910,-1.887376,-3.047510,9.129493,-0.880685,2.032404,-13.127419,0.156240,4.542413,4.312793,...,0.345739,0.057034,-0.583898,0.379561,0.059571,0.227318,0.444394,-0.437739,-0.166733,-0.312971
860,2.842940,-2.805740,-3.121849,7.156009,-0.751311,1.157776,-9.784992,-1.171517,4.927120,3.776248,...,0.145041,-0.117903,-0.560734,-0.329986,0.331400,0.219667,0.757997,-0.191230,-0.352478,-0.287249
861,3.543118,-5.118462,-5.529815,11.363193,-1.617810,-0.448854,-17.443207,-1.281686,11.976348,6.931947,...,0.213534,-0.054093,-0.417278,-0.043411,0.111895,0.549025,0.621648,-0.193246,0.065221,-0.679957
862,2.673864,-8.289579,-2.439484,9.520558,2.216386,0.929411,-17.339121,-3.470526,7.684749,1.354805,...,0.153356,0.322790,0.172389,-0.212086,0.009056,-0.047872,0.381528,-0.032149,0.240377,-0.631686


(864, 5374)

(864, 4200)

(864, 3076)

In [3]:
for i in range(10, 11):
    X1_all = load_datasets(f"A2AR/data/a2ar_train_{i}")

    X2_all = load_datasets(f"A2AR/data/a2ar_val_{i}")

    X3_all = load_datasets(f"A2AR/data/a2ar_test_{i}")
    cls = Dataset_creator(['Mol2Vec', 'RoBERTa_ZINC'])
    X1_all = cls.fit_transform(X1_all)
    X2_all = cls.transform(X2_all)
    X3_all = cls.transform(X3_all)

    X1_all.X.to_csv(f"A2AR/mod_data/X1.{i}")
    X2_all.X.to_csv(f"A2AR/mod_data/X2.{i}")
    X3_all.X.to_csv(f"A2AR/mod_data/X3.{i}")
    X1_all.y.to_csv(f"A2AR/mod_data/y1.{i}")
    X2_all.y.to_csv(f"A2AR/mod_data/y2.{i}")
    X3_all.y.to_csv(f"A2AR/mod_data/y3.{i}")

(2398, 4306)

pandas.core.frame.DataFrame

Some weights of RobertaModel were not initialized from the model checkpoint at entropy/roberta_zinc_480m and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


pandas.core.frame.DataFrame

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,3.031286,-3.454808,-2.263860,6.271306,2.995978,0.981784,-10.757834,-1.355217,6.775432,-1.027848,...,0.016533,0.195609,-0.674374,-0.065700,0.370229,-0.326461,0.348396,-0.086251,0.064743,-0.962521
1,3.350501,-1.980245,-3.243645,4.621302,-2.423914,-0.357021,-12.652489,0.289971,5.873908,4.563121,...,0.066493,0.313761,-0.687266,-0.086003,0.015799,0.499730,0.417170,-0.433060,0.025140,-0.010341
2,3.714069,-5.852489,-2.354247,9.809155,-0.262454,1.374667,-12.131661,-2.852122,5.135067,0.854150,...,0.427338,-0.119034,-0.220593,-0.327747,0.018900,-0.180831,0.066138,-0.359251,0.187227,-0.331075
3,3.081486,-8.096527,-2.389479,9.637930,1.337607,0.428647,-15.852401,-2.957059,7.091984,2.542176,...,0.212281,0.368771,0.153928,-0.228844,0.012988,-0.085077,0.429632,-0.064018,0.113950,-0.692819
4,0.723944,-4.291038,-3.183181,8.225389,0.743683,1.325434,-11.569109,-2.816562,5.148249,0.316404,...,0.240745,-0.182091,-0.124104,0.135646,-0.196018,0.062310,0.405304,-0.025201,-0.037708,-0.288250
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2393,2.653754,-7.146047,-2.449787,9.172974,1.035106,0.775873,-14.914745,-2.719624,8.021161,2.170897,...,0.448256,0.491986,-0.357603,-0.081183,-0.278233,-0.135188,0.569269,-0.283329,-0.071396,-0.738980
2394,0.854948,-4.229382,-4.969607,9.971701,0.328093,0.074672,-16.193737,-2.687413,11.379488,4.477886,...,0.270919,-0.161545,-0.195868,-0.026008,0.102916,0.140246,0.669294,-0.233838,0.130424,-0.477552
2395,3.666460,-3.424716,-4.983419,9.952207,1.400621,0.672362,-12.883733,-3.339542,9.537730,5.429902,...,0.045195,0.249221,-0.354409,0.071646,0.125062,0.288560,0.914358,-0.259063,0.250934,-0.480186
2396,3.461942,-5.417378,-3.468525,7.684143,1.368830,-0.217005,-13.005549,-3.189668,8.138423,2.356131,...,0.448488,-0.177408,-0.629834,-0.152293,-0.186566,0.062021,0.700484,-0.078713,0.170892,-0.848403


...

pandas.core.frame.DataFrame

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,4.719957,-6.825691,-5.275553,7.427641,2.117555,-4.409114,-20.588449,-4.263409,9.039432,1.124004,...,-0.117531,0.066307,0.073473,0.053947,-0.011456,0.248302,0.882871,-0.134034,0.101105,-0.926051
1,2.676739,-4.529388,-2.503095,8.664835,-1.422678,1.718229,-10.841211,-0.888626,5.250077,1.132984,...,0.361555,-0.109810,-0.367811,0.251796,-0.052785,0.133834,0.889266,-0.269218,0.163290,-0.007741
2,2.528946,-2.400130,-2.897563,7.094854,-0.478954,0.433469,-10.049082,-1.011271,4.869930,3.022278,...,0.089385,-0.039821,-0.385214,-0.412184,0.306683,0.209005,0.888776,-0.416939,-0.295426,-0.088778
3,4.035905,-2.917027,-3.177843,4.870048,0.714673,0.467616,-10.627448,-1.463339,2.508328,3.341819,...,-0.051249,0.019601,-0.234947,-0.663630,-0.225576,0.667409,0.739919,-0.373016,-0.551484,-0.496429
4,6.791410,-9.079959,-3.285735,11.249070,0.780106,-0.025624,-16.080940,-2.889991,5.756549,3.822708,...,-0.162695,0.212831,-0.220516,0.046122,-0.050340,0.350902,0.288819,-0.058950,-0.008380,-0.778272
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
896,0.393037,1.031070,-2.772204,3.237898,0.252583,1.224730,-6.474216,1.577231,4.122285,-1.307249,...,-0.373801,0.126145,-0.627233,-0.206186,-0.316600,0.039116,1.104801,-0.250967,-0.252664,-0.419314
897,5.456210,-7.532487,-3.614988,4.579227,-2.215615,-3.747138,-11.761658,-1.178334,3.789686,1.231824,...,0.045980,0.137823,0.132906,0.014447,0.027588,0.099928,0.680277,-0.119590,0.073923,-0.639346
898,5.055686,-4.835310,-3.041866,9.995363,-2.668548,0.958239,-12.736185,-1.545098,12.372886,4.902917,...,0.525436,-0.305444,-0.077900,-0.097706,0.587755,0.097803,0.859146,-0.597544,0.245550,-0.272907
899,2.842940,-2.805740,-3.121849,7.156009,-0.751311,1.157776,-9.784992,-1.171517,4.927120,3.776248,...,0.145041,-0.117903,-0.560734,-0.329986,0.331400,0.219667,0.757997,-0.191230,-0.352478,-0.287249


(901, 5374)

(901, 4259)

(901, 3169)

In [ ]:
X1_all.X.to_csv("A2AR/mod_data/X1.1")
X2_all.X.to_csv("A2AR/mod_data/X2.1")
X3_all.X.to_csv("A2AR/mod_data/X3.1")
X1_all.y.to_csv("A2AR/mod_data/y1.1")
X2_all.y.to_csv("A2AR/mod_data/y2.1")
X3_all.y.to_csv("A2AR/mod_data/y3.1")

In [11]:
X1_all.getDF()

,QSPRID,Y,Drug,Y_original
QSPRID,,,,
A2ARDataset_0000,A2ARDataset_0000,True,Cc1cc(C)n(-c2cc(NC(=O)CCN(C)C)nc(-c3ccc(C)o3)n...,True
A2ARDataset_0001,A2ARDataset_0001,True,CNC(=O)C12CC1C(n1cnc3c(NCc4cccc(Cl)c4)nc(C#CCC...,True
A2ARDataset_0002,A2ARDataset_0002,False,COc1nc(N)c(C#N)c(-c2ccc3c(c2)OCO3)c1C#N,False
A2ARDataset_0003,A2ARDataset_0003,True,CCNC(=O)C1OC(n2cnc3c(NCC)nc(C#CCCCc4ccccc4)nc3...,True
A2ARDataset_0004,A2ARDataset_0004,True,Cc1cc(C)n(-c2cc(NC(=O)CN3CCOCC3)nc(-c3ccc(C)o3...,True
...,...,...,...,...
A2ARDataset_2402,A2ARDataset_2402,True,CCCn1cc2c(nc(NC(=O)Nc3ccccc3OC)n3nc(-c4ccco4)n...,True
A2ARDataset_2403,A2ARDataset_2403,True,O=C(COc1ccc(-c2cc3c([nH]2)c(=O)n(CC2CC2)c(=O)n...,True
A2ARDataset_2404,A2ARDataset_2404,True,CNc1ncc(C(=O)NCc2ccc(OC)cc2)c2nc(-c3ccco3)nn12,True
